In [1]:
import os
import pandas as pd
import numpy as np

## Build clusters, leakage analysis & cluster-aware split

Fixes from baseline notebooks:
- `edge_mode` variable was used but never defined (NameError)
- Cluster-aware split now keeps entire clusters in the same split (greedy bin-packing)
  instead of distributing individual clustered images, which caused the very leakage it was meant to prevent

In [2]:
noisy_data_path = "/srv/defectDetectionDataset/surfaceClassification/noisy"
artifacts_path = "./artifacts/noisy"

df = pd.read_csv(os.path.join(artifacts_path, "noisy_images.csv"))
similarity_df = pd.read_csv(os.path.join(artifacts_path, "similarity_pairs.csv"))

print(f"Images: {len(df)}, Similarity pairs: {len(similarity_df)}")

Images: 5475, Similarity pairs: 71204


## Build clusters (clique-style growth + connected components)

In [5]:
thr = 0.92  # reuse threshold from baseline analysis

edges_df = similarity_df[similarity_df["similarity"] >= thr][
    ["image_id_1", "image_id_2", "similarity"]
].copy()

print(f"Threshold: {thr}")
print(f"Edges (similarity >= threshold): {len(edges_df):,}")

# Build symmetric similarity lookup
sim_map = {}
for r in edges_df.itertuples(index=False):
    a = str(r.image_id_1)
    b = str(r.image_id_2)
    if a == b:
        continue
    key = (a, b) if a <= b else (b, a)
    sim = float(r.similarity)
    if key not in sim_map or sim > sim_map[key]:
        sim_map[key] = sim

neighbors = {}
for (a, b), sim in sim_map.items():
    neighbors.setdefault(a, set()).add(b)
    neighbors.setdefault(b, set()).add(a)

# Clique-style growth: a node joins a cluster only if similar to ALL members
nodes = sorted(neighbors.keys(), key=lambda n: len(neighbors[n]), reverse=True)
clusters = []
for node in nodes:
    placed = False
    for cluster in clusters:
        ok = True
        for member in cluster:
            key = (node, member) if node <= member else (member, node)
            sim = sim_map.get(key)
            if sim is None or sim < thr:
                ok = False
                break
        if ok:
            cluster.append(node)
            placed = True
            break
    if not placed:
        clusters.append([node])

clusters = [sorted(c) for c in clusters if len(c) >= 2]
print(f"Clique clusters (size >= 2): {len(clusters):,}")

# Connected components via BFS
edge_list = []
for cluster in clusters:
    for i in range(len(cluster)):
        for j in range(i + 1, len(cluster)):
            key = (cluster[i], cluster[j]) if cluster[i] <= cluster[j] else (cluster[j], cluster[i])
            sim = sim_map.get(key)
            if sim is not None and sim >= thr:
                edge_list.append((key[0], key[1], sim))

graph = {}
for a, b, sim in edge_list:
    graph.setdefault(a, set()).add(b)
    graph.setdefault(b, set()).add(a)

visited = set()
final_clusters = []
for node in graph:
    if node in visited:
        continue
    stack = [node]
    component = []
    visited.add(node)
    while stack:
        cur = stack.pop()
        component.append(cur)
        for nbr in graph[cur]:
            if nbr not in visited:
                visited.add(nbr)
                stack.append(nbr)
    if len(component) >= 2:
        final_clusters.append(sorted(component))

final_clusters.sort(key=len, reverse=True)
print(f"Clusters (size >= 2): {len(final_clusters):,}")
if final_clusters:
    print(f"Largest cluster size: {len(final_clusters[0])}")

# Build membership table
membership_rows = []
for cid, members in enumerate(final_clusters):
    for image_id in members:
        membership_rows.append({"cluster_id": cid, "image_id": image_id, "threshold": thr})

clusters_df = pd.DataFrame(membership_rows)
if not df.empty and not clusters_df.empty:
    clusters_df = clusters_df.merge(df, on="image_id", how="left")

out_path = os.path.join(artifacts_path, f"near_duplicate_clusters_thr_{thr:.3f}.csv")
clusters_df.to_csv(out_path, index=False)
print(f"Saved cluster membership to: {out_path}")

Threshold: 0.92
Edges (similarity >= threshold): 4,181
Clique clusters (size >= 2): 938
Clusters (size >= 2): 938
Largest cluster size: 11
Saved cluster membership to: ./artifacts/noisy/near_duplicate_clusters_thr_0.920.csv


## Data leakage analysis (naive random split)

In [6]:
import torch
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

ds = ImageFolder(noisy_data_path)

# Build index -> image_id mapping from the dataset sample paths
index_to_image_id = {}
for idx, (path, _label) in enumerate(ds.samples):
    index_to_image_id[idx] = os.path.basename(path)

cluster_map = {str(r.image_id): r.cluster_id for r in clusters_df.itertuples(index=False)}

n_train = int(0.8 * len(ds))
n_val = len(ds) - n_train

train_leak_rates = []
val_leak_rates = []

for i in range(5):
    seed = 42 + i
    g = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(ds, [n_train, n_val], generator=g)

    train_ids = {index_to_image_id[i] for i in train_subset.indices}
    val_ids = {index_to_image_id[i] for i in val_subset.indices}

    train_clusters = {cluster_map.get(img_id) for img_id in train_ids if img_id in cluster_map}
    val_clusters = {cluster_map.get(img_id) for img_id in val_ids if img_id in cluster_map}
    train_clusters.discard(None)
    val_clusters.discard(None)

    n_train_leak = sum(1 for img_id in train_ids if cluster_map.get(img_id) in val_clusters)
    n_val_leak = sum(1 for img_id in val_ids if cluster_map.get(img_id) in train_clusters)

    train_rate = n_train_leak / len(train_ids) if train_ids else 0.0
    val_rate = n_val_leak / len(val_ids) if val_ids else 0.0

    train_leak_rates.append(train_rate)
    val_leak_rates.append(val_rate)

    print(
        f"Seed {seed}: train->val leakage {n_train_leak}/{len(train_ids)} ({train_rate:.2%}), "
        f"val<-train leakage {n_val_leak}/{len(val_ids)} ({val_rate:.2%})"
    )

print(f"\nMean train->val leakage: {np.mean(train_leak_rates):.2%}")
print(f"Mean val<-train leakage: {np.mean(val_leak_rates):.2%}")

Seed 42: train->val leakage 669/4380 (15.27%), val<-train leakage 413/1095 (37.72%)
Seed 43: train->val leakage 662/4380 (15.11%), val<-train leakage 442/1095 (40.37%)
Seed 44: train->val leakage 680/4380 (15.53%), val<-train leakage 425/1095 (38.81%)
Seed 45: train->val leakage 628/4380 (14.34%), val<-train leakage 430/1095 (39.27%)
Seed 46: train->val leakage 617/4380 (14.09%), val<-train leakage 404/1095 (36.89%)

Mean train->val leakage: 14.87%
Mean val<-train leakage: 38.61%


## Cluster-aware split (70/15/15)

**Key fix**: entire clusters are assigned to the same split via greedy bin-packing.
The baseline code distributed individual images from clusters across splits,
which did NOT prevent leakage.

In [7]:
all_ids = df["image_id"].astype(str).unique().tolist()
cluster_map = {str(r.image_id): int(r.cluster_id) for r in clusters_df.itertuples(index=False)}

# Group images: clustered images share a group, singletons are individual groups
cluster_to_images = {}
for img_id in all_ids:
    cid = cluster_map.get(img_id)
    if cid is not None:
        cluster_to_images.setdefault(cid, []).append(img_id)

clustered_set = set(cluster_map.keys())
singleton_ids = [img_id for img_id in all_ids if img_id not in clustered_set]

# Build groups: one per cluster + one per singleton
groups = list(cluster_to_images.values()) + [[s] for s in singleton_ids]

# Shuffle then sort largest-first (stable sort keeps random order for same-size groups)
rng = np.random.default_rng(42)
rng.shuffle(groups)
groups.sort(key=len, reverse=True)

n_total = len(all_ids)
targets = {"train": 0.70 * n_total, "val": 0.15 * n_total, "test": 0.15 * n_total}
current = {"train": 0, "val": 0, "test": 0}
split_assignment = {}  # image_id -> split

# Greedy: assign each group to the split with the largest remaining capacity
for group in groups:
    best_split = max(targets, key=lambda s: targets[s] - current[s])
    for img_id in group:
        split_assignment[img_id] = best_split
    current[best_split] += len(group)

# Build split DataFrame
split_rows = [{"image_id": img_id, "split": split} for img_id, split in split_assignment.items()]
split_df = pd.DataFrame(split_rows)
split_df = split_df.merge(df, on="image_id", how="left")

out_path = os.path.join(artifacts_path, "cluster_aware_split_70_15_15.csv")
split_df.to_csv(out_path, index=False)

# Report
for split in ["train", "val", "test"]:
    sub = split_df[split_df["split"] == split]
    n_clustered = sub["image_id"].isin(clustered_set).sum()
    print(f"{split}: {len(sub)} images ({n_clustered} clustered, {len(sub) - n_clustered} singletons)")

print(f"\nSaved split manifest to {out_path}")

train: 3833 images (2409 clustered, 1424 singletons)
val: 821 images (0 clustered, 821 singletons)
test: 821 images (0 clustered, 821 singletons)

Saved split manifest to ./artifacts/noisy/cluster_aware_split_70_15_15.csv


## Verify zero leakage in cluster-aware split

In [8]:
# Verify that no cluster is split across train/val/test
split_map = dict(zip(split_df["image_id"].astype(str), split_df["split"]))

leaking_clusters = 0
for cid, members in cluster_to_images.items():
    splits_in_cluster = {split_map.get(m) for m in members}
    splits_in_cluster.discard(None)
    if len(splits_in_cluster) > 1:
        leaking_clusters += 1
        print(f"  Cluster {cid}: splits = {splits_in_cluster}, members = {members[:5]}...")

if leaking_clusters == 0:
    print("No cluster is split across train/val/test -- zero leakage.")
else:
    print(f"WARNING: {leaking_clusters} clusters leak across splits!")

No cluster is split across train/val/test -- zero leakage.


## Copy images into split folders

In [9]:
from pathlib import Path
from shutil import copy2

out_root = Path("/srv/defectDetectionDataset/surfaceClassification/noisy_clustered")

for split in ["train", "val", "test"]:
    (out_root / split).mkdir(parents=True, exist_ok=True)

copied = 0
missing = 0
for row in split_df.itertuples(index=False):
    src = Path(noisy_data_path + row.file_path)
    if not src.exists():
        missing += 1
        continue
    dest_dir = out_root / row.split / row.label
    dest_dir.mkdir(parents=True, exist_ok=True)
    copy2(src, dest_dir / src.name)
    copied += 1

print(f"Copied {copied} images to {out_root}")
if missing:
    print(f"Skipped {missing} missing files")

Copied 5475 images to /srv/defectDetectionDataset/surfaceClassification/noisy_clustered
